# Plan Phase H (= v5): 距離重み + snapshot filter で value 学習を強化 (Colab T4 GPU)

v4 (= naive value 学習) の問題:
- 序盤 snapshot (= turn 1-3) にも target = final_winner ±1 を割り当て → ノイズ大
- NN が「序盤 position は意味なし」 と学習 → 守備的に偏る
- 直接対戦 (= NN-on vs NN-off) で NN 0% 勝率

v5 の修正:
- **discounted target**: y = final_winner × γ^(end_turn - current_turn)、 γ=0.9
  - turn 1 snapshot: y ≈ ±0.3 (= 弱い信号)
  - turn 13 snapshot: y ≈ ±1.0 (= 強い信号)
- **snapshot filter**: turn >= 4 だけ学習 (= 0-3 はほぼ random、 学習に害)
- 既存 db/self_play_snapshots_v2.jsonl 流用、 Drive アップロード不要


## 1. GPU 確認

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
assert torch.cuda.is_available(), 'GPU が有効化されていません。 ランタイム > T4 GPU を選択してください。'

## 2. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAPSHOT_PATH = os.path.join(WORK_DIR, 'self_play_snapshots_v2.jsonl')  # v2 流用 (= 既に Drive にあるはず)
OUTPUT_PT = os.path.join(WORK_DIR, 'nn_eval_v5.pt')
OUTPUT_CURVE = os.path.join(WORK_DIR, 'training_curve_v5.json')

assert os.path.exists(SNAPSHOT_PATH), f'snapshot v2 が見つからない: {SNAPSHOT_PATH}'
print('snapshot size:', os.path.getsize(SNAPSHOT_PATH) // (1024*1024), 'MB')


## 3. snapshot ロード + tensor 化 (= 172 dim state_encoded + action_idx)

メモリに注意: 30 万 snap × 172 float = 約 200MB、 GPU 12GB に余裕で乗る。

In [ ]:
import json, time
import numpy as np

N_ACTION_CATEGORIES = 9
STATE_DIM = 172

t0 = time.time()
snapshots = []
with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            snap = json.loads(line)
        except json.JSONDecodeError:
            continue
        # state_encoded が無い snapshot は skip (= v1 互換 fallback)
        if 'state_encoded' not in snap or 'action_idx' not in snap:
            continue
        snapshots.append(snap)
print(f'loaded {len(snapshots):,} v2 snapshots in {time.time()-t0:.1f}s')

# action_idx 分布 + policy 学習可能性チェック
from collections import Counter
ac = Counter(s['action_idx'] for s in snapshots)
print(f'action_idx distribution: {sorted(ac.items())}')
print(f'unknown (= -1) ratio: {ac.get(-1, 0) / len(snapshots):.1%}')

In [ ]:
# v5: snapshot filter (turn>=4) + discounted target (γ^Δturn)
GAMMA = 0.9
MIN_TURN = 4

# action_idx = -1 (= unknown) の snapshot は除外
snapshots = [s for s in snapshots if s.get("action_idx", -1) >= 0]
print(f"after action_idx filter: {len(snapshots):,}")

# turn >= MIN_TURN で filter
snapshots = [s for s in snapshots if s.get("turn", 0) >= MIN_TURN]
print(f"after turn >= {MIN_TURN} filter: {len(snapshots):,}")

# game_idx ごとの max turn を計算 (= discount で使用)
from collections import defaultdict
game_max_turn = defaultdict(int)
for s in snapshots:
    gi = s.get("game_idx")
    if gi is not None:
        game_max_turn[gi] = max(game_max_turn[gi], s.get("turn", 0))
print(f"unique games (= post filter): {len(game_max_turn)}")
print(f"game max_turn distribution (sample 5): {list(game_max_turn.items())[:5]}")

device = torch.device("cuda")

t0 = time.time()
X_np = np.zeros((len(snapshots), STATE_DIM), dtype=np.float32)
y_np = np.zeros((len(snapshots),), dtype=np.float32)
a_np = np.zeros((len(snapshots),), dtype=np.int64)
w_np = np.ones((len(snapshots),), dtype=np.float32)  # sample weight (= discount magnitude)

for i, snap in enumerate(snapshots):
    se = snap["state_encoded"]
    if len(se) >= STATE_DIM:
        X_np[i, :] = se[:STATE_DIM]
    else:
        X_np[i, :len(se)] = se

    final = float(snap.get("final_winner", 0))
    cur_turn = snap.get("turn", 0)
    gi = snap.get("game_idx")
    end_turn = game_max_turn.get(gi, cur_turn) if gi is not None else cur_turn
    dist = max(0, end_turn - cur_turn)
    discount = GAMMA ** dist
    y_np[i] = final * discount  # discounted target
    a_np[i] = int(snap["action_idx"])
    w_np[i] = discount  # value loss weight 用 (= 序盤 snapshot は重み低)
print(f"tensor build: {time.time()-t0:.1f}s")

print(f"y_np stats: mean={y_np.mean():.3f}, std={y_np.std():.3f}, min={y_np.min():.3f}, max={y_np.max():.3f}")
print(f"w_np stats: mean={w_np.mean():.3f}, min={w_np.min():.3f}, max={w_np.max():.3f}")

X = torch.from_numpy(X_np).to(device)
y = torch.from_numpy(y_np).unsqueeze(1).to(device)
a = torch.from_numpy(a_np).to(device)
w = torch.from_numpy(w_np).unsqueeze(1).to(device)

n = len(snapshots)
split = int(n * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
a_train, a_val = a[:split], a[split:]
w_train, w_val = w[:split], w[split:]
print(f"train: {X_train.shape[0]:,}, val: {X_val.shape[0]:,}, dim: {STATE_DIM}")
del snapshots, X_np, y_np, a_np, w_np


## 4. モデル定義 (= StateEncoderEvaluator、 172 → 256 → 128 → 64)

In [ ]:
import torch.nn as nn

class StateEncoderEvaluator(nn.Module):
    """Phase G: 172 dim state_encoded を直接入力。 policy + value 両頭を本気で学習。"""
    def __init__(self, input_dim=172, dropout=0.2):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.value_head = nn.Linear(64, 1)
        self.policy_head = nn.Linear(64, N_ACTION_CATEGORIES)
    def forward(self, x):
        h = self.shared(x)
        return self.value_head(h), self.policy_head(h)

model = StateEncoderEvaluator(input_dim=STATE_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'params: {n_params:,}')

## 5. 学習 (= value + policy 双方を本気で学習)

- value loss = MSE (= final_winner ±1)
- policy loss = CrossEntropy (= action_idx 0-8)
- 複合 loss = value_loss + **1.0** × policy_loss (= v1-v3 の 0.5 から policy 重視に変更)
- val metric: value sign acc + **action acc** 両方追跡

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 100
BATCH = 1024
LR_MAX = 1e-3
LR_MIN = 1e-5
EARLY_STOP_PATIENCE = 15
POLICY_WEIGHT = 1.0  # v1-v3 の 0.5 から強化

train_ds = TensorDataset(X_train, y_train, a_train, w_train)  # v5: 重み付き
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=LR_MAX)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
value_criterion = nn.MSELoss()
policy_criterion = nn.CrossEntropyLoss()

# best 判定: value_sign_acc + action_acc の平均で決める (= 両方良いものを保存)
best_combined = 0.0
# v5: NOTE - weighted MSE で序盤 snapshot の noise を抑制
best_val_sign_acc = 0.0
best_action_acc = 0.0
epochs_since_best = 0
curve = []

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_loss_v_sum = 0.0
    train_loss_p_sum = 0.0
    n_batches = 0
    for Xb, yb, ab, wb in train_loader:  # v5: weighted
        optimizer.zero_grad()
        v_pred, p_pred = model(Xb)
        loss_v = ((v_pred - yb) ** 2 * wb).mean()  # v5: weighted MSE (= sample weight)
        loss_p = policy_criterion(p_pred, ab)
        loss = loss_v + POLICY_WEIGHT * loss_p
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        train_loss_v_sum += loss_v.item()
        train_loss_p_sum += loss_p.item()
        n_batches += 1

    model.eval()
    with torch.no_grad():
        v_pred_val, p_pred_val = model(X_val)
        sign_acc = float((torch.sign(v_pred_val) == torch.sign(y_val)).float().mean().item())
        action_acc = float((p_pred_val.argmax(dim=1) == a_val).float().mean().item())
        combined = (sign_acc + action_acc) / 2

    avg_loss = train_loss_sum / max(1, n_batches)
    avg_loss_v = train_loss_v_sum / max(1, n_batches)
    avg_loss_p = train_loss_p_sum / max(1, n_batches)
    current_lr = optimizer.param_groups[0]['lr']
    curve.append({
        'epoch': epoch+1,
        'train_loss': avg_loss,
        'train_loss_value': avg_loss_v,
        'train_loss_policy': avg_loss_p,
        'val_sign_acc': sign_acc,
        'val_action_acc': action_acc,
        'lr': current_lr,
    })
    msg = (f'ep {epoch+1:3d}/{EPOCHS}: loss={avg_loss:.3f} (v={avg_loss_v:.3f},p={avg_loss_p:.3f}), '
           f'sign_acc={sign_acc:.3f}, action_acc={action_acc:.3f}, combined={combined:.3f}, lr={current_lr:.2e}')

    if combined > best_combined:
        best_combined = combined
        best_val_sign_acc = sign_acc
        best_action_acc = action_acc
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, OUTPUT_PT)
        epochs_since_best = 0
        msg += f'  ✔ saved (combined={combined:.3f})'
    else:
        epochs_since_best += 1
    print(msg)

    scheduler.step()

    if epochs_since_best >= EARLY_STOP_PATIENCE:
        print(f'\n=== early stopping at epoch {epoch+1} ===')
        break

with open(OUTPUT_CURVE, 'w', encoding='utf-8') as f:
    json.dump(curve, f, ensure_ascii=False, indent=2)
print(f'\n=== DONE ===')
print(f'best combined: {best_combined:.3f}')
print(f'  best val_sign_acc: {best_val_sign_acc:.3f}')
print(f'  best val_action_acc: {best_action_acc:.3f}')
print(f'v3 sign_acc baseline: ?')

## 6. 成果物確認 + 学習曲線可視化

In [ ]:
import matplotlib.pyplot as plt

for p in [OUTPUT_PT, OUTPUT_CURVE]:
    if os.path.exists(p):
        print(f'  {p}: {os.path.getsize(p):,} bytes')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_x = [c['epoch'] for c in curve]
axes[0].plot(epochs_x, [c['train_loss_value'] for c in curve], label='value loss')
axes[0].plot(epochs_x, [c['train_loss_policy'] for c in curve], label='policy loss')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].set_yscale('log')
axes[1].plot(epochs_x, [c['val_sign_acc'] for c in curve], label='val sign acc', color='blue')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='coin flip')
axes[1].axhline(best_val_sign_acc, color='green', linestyle='--', alpha=0.5, label=f'best={best_val_sign_acc:.3f}')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('value sign acc'); axes[1].legend()
axes[2].plot(epochs_x, [c['val_action_acc'] for c in curve], label='val action acc', color='orange')
axes[2].axhline(1/9, color='gray', linestyle='--', alpha=0.5, label='random 1/9=0.11')
axes[2].axhline(best_action_acc, color='green', linestyle='--', alpha=0.5, label=f'best={best_action_acc:.3f}')
axes[2].set_xlabel('epoch'); axes[2].set_ylabel('action acc'); axes[2].legend()
plt.tight_layout(); plt.show()

print('\n--- 次の手順 ---')
print('1. Drive の nn_eval_v4.pt と training_curve_v4.json の共有 URL を取得')
print('2. Claude に貼り付け')
print('3. engine 統合は別 task (= engine.nn_eval に StateEncoderEvaluator 対応を追加)')